# Solutions — Joins Deep Dive (Medium 14)

**Datasets:** `bakehouse` tables, `tpch` tables

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

customers    = spark.table("samples.bakehouse.sales_customers")
transactions = spark.table("samples.bakehouse.sales_transactions")
tpch_customer = spark.table("samples.tpch.customer")
tpch_nation   = spark.table("samples.tpch.nation")

## Solution 1 — Inner vs Left Join Row Count Comparison

In [ ]:
inner_cnt = customers.join(transactions, on="customerID", how="inner").count()
left_cnt  = customers.join(transactions, on="customerID", how="left").count()
result_1 = spark.createDataFrame(
    [("inner", inner_cnt), ("left", left_cnt)],
    ["join_type","row_count"]
)

## Solution 2 — Left Semi-Join (Customers with Transactions)

In [ ]:
# Why: semi-join keeps only left-side columns and eliminates duplicates
result_2 = (
    customers
    .join(transactions.select("customerID").distinct(), on="customerID", how="left_semi")
    .select("customerID","first_name","last_name","country")
)

## Solution 3 — Left Anti-Join (Customers Without Transactions)

In [ ]:
result_3 = (
    customers
    .join(transactions.select("customerID").distinct(), on="customerID", how="left_anti")
    .select("customerID","first_name","last_name","country")
)

## Solution 4 — Full Outer Join: Unmatched Customers and Transactions

In [ ]:
joined = customers.join(transactions, on="customerID", how="full_outer")
customers_no_txn = joined.filter(F.col("transactionID").isNull()).count()
txns_no_customer = joined.filter(F.col("first_name").isNull()).count()
result_4 = spark.createDataFrame(
    [("customers_without_transactions", customers_no_txn),
     ("transactions_without_customers", txns_no_customer)],
    ["category","count"]
)

## Solution 5 — Broadcast Hint for Small Table Join

In [ ]:
from pyspark.sql.functions import broadcast
result_5 = (
    tpch_customer
    .join(broadcast(tpch_nation),
          tpch_customer["c_nationkey"] == tpch_nation["n_nationkey"])
    .select("c_name","c_mktsegment","n_name")
    .limit(1000)
)

## Solution 6 — Handle Duplicate Column Names After Join

In [ ]:
# Why: aliasing the tables allows unambiguous column references after join
cust = customers.alias("c")
txn  = transactions.alias("t")
result_6 = (
    txn
    .join(cust, on="customerID")
    .select(
        F.col("t.transactionID"),
        F.col("c.city").alias("customer_city"),
        F.col("t.product"),
        F.col("t.totalPrice")
    )
)

## Solution 7 — Cross Join (Regions × Quarters)

In [ ]:
regions  = tpch_nation.select("n_name").distinct().limit(5).toDF("r_name")
quarters = spark.createDataFrame([("Q1",),("Q2",),("Q3",),("Q4",)], ["quarter"])
result_7 = regions.crossJoin(quarters).orderBy("r_name","quarter")